# Credit Risk Modeling in Fixed Income: Theory & Implementation

This notebook serves as an academic companion to the theoretical foundations of credit risk in a fixed income context. We bridge the gap between financial mathematics, stochastic calculus, and quantitative implementation across three core domains:
1. **Structural Models** (Merton's framework based on firm value)
2. **Reduced-Form Models & CDS** (Intensity-based pricing using Poisson processes)
3. **Credit Valuation Adjustment (CVA)** (Counterparty risk pricing)


## 1. Credit Analysis: The Merton Structural Model

Pioneered by Robert Merton (1974), structural models operate on the premise that default is triggered endogenously. Specifically, default occurs at maturity $T$ if the firm's asset value $V_T$ falls below the face value of its outstanding zero-coupon debt $D$.

### Asset Dynamics
We assume the firm's asset value follows a Geometric Brownian Motion (GBM) under the physical probability measure $\mathbb{P}$:
$$dV_t = \mu V_t dt + \sigma_V V_t dW_t$$

Because limited liability protects equity holders, they cannot lose more than their investment. Equity can therefore be modeled as a European call option on the firm's assets $V_t$ with a strike price equal to the debt $D$. Using the Black-Scholes-Merton framework under the risk-neutral measure $\mathbb{Q}$ (where the drift $\mu$ is replaced by the risk-free rate $r$):

$$E_0 = V_0 \Phi(d_1) - e^{-rT} D \Phi(d_2)$$
$$d_1 = \frac{\ln(V_0/D) + (r + \frac{1}{2}\sigma_V^2)T}{\sigma_V \sqrt{T}}, \quad d_2 = d_1 - \sigma_V \sqrt{T}$$

### Distance to Default & Probability of Default
The **Distance to Default (DD)** quantifies how many standard deviations the expected asset value is away from the default barrier. Because actual default is an empirical event, $DD$ is strictly evaluated under the physical measure $\mathbb{P}$ using the true asset drift $\mu$:

$$DD = \frac{\ln(V_0/D) + (\mu - \frac{1}{2}\sigma_V^2)T}{\sigma_V \sqrt{T}}$$

The physical Probability of Default (PD) is then:
$$PD = \mathbb{P}(V_T < D) = \Phi(-DD)$$


In [ ]:
import numpy as np
from scipy.stats import norm

class MertonModel:
    def __init__(self, V, D, T, r, sigma_V, mu=None):
        self.V = V
        self.D = D
        self.T = T
        self.r = r
        self.sigma_V = sigma_V
        # Use physical drift if provided, otherwise default to risk-neutral rate
        self.mu = mu if mu is not None else r 

    def _d_terms(self, use_mu=False):
        drift = self.mu if use_mu else self.r
        d1 = (np.log(self.V / self.D) + (drift + 0.5 * self.sigma_V**2) * self.T) / (self.sigma_V * np.sqrt(self.T))
        d2 = d1 - self.sigma_V * np.sqrt(self.T)
        return d1, d2

    def equity_value(self):
        d1, d2 = self._d_terms(use_mu=False)
        E = self.V * norm.cdf(d1) - np.exp(-self.r * self.T) * self.D * norm.cdf(d2)
        return E

    def distance_to_default(self):
        # DD is calculated using the physical measure (mu)
        _, d2 = self._d_terms(use_mu=True)
        return d2 

    def probability_of_default(self):
        return norm.cdf(-self.distance_to_default())

# Application Example
model = MertonModel(V=100.0, D=80.0, T=1.0, r=0.05, sigma_V=0.20, mu=0.10)
print(f"Equity Value (E): {model.equity_value():.4f}")
print(f"Distance to Default (DD): {model.distance_to_default():.4f}")
print(f"Probability of Default (PD): {model.probability_of_default():.4%}")

## 2. Reduced-Form Models and the Hazard Rate ($\lambda$)

While structural models are intuitive, they often fail to capture short-term credit spreads accurately because firm value is rarely observable continuously. **Reduced-form models** (e.g., Jarrow-Turnbull, Duffie-Singleton) treat default not as an endogenous barrier event, but as an exogenous stochastic process.

### The Default Intensity
Default is modeled as the first arrival time $\tau$ of a Poisson process with an intensity or **hazard rate** $\lambda(t)$. The hazard rate represents the instantaneous probability of default occurring in the next infinitesimal time interval $dt$, conditional on having survived up to time $t$:

$$\mathbb{P}(t < \tau \le t + dt \mid \tau > t) \approx \lambda(t) dt$$

The **Survival Probability** $S(t)$, which is the probability that the issuer survives until time $t$ (i.e., $\tau > t$), is given by:
$$S(t) = \mathbb{E}^\mathbb{Q}\left[ \exp\left(-\int_0^t \lambda(u) du\right) \right]$$

If we assume a deterministic and constant hazard rate $\lambda$, the survival curve simplifies to an exponential decay:
$$S(t) = e^{-\lambda t}$$


## 3. Credit Default Swaps (CDS) Valuation

A CDS is a derivative contract that transfers the credit exposure of fixed income products between parties. It is priced by equating the expected present value of the premium leg (payments made by the protection buyer) to the expected present value of the protection leg (contingent payment made by the protection seller upon default).

The fair or par CDS spread $s^*$ sets the net present value to zero:
$$s^* = \frac{PV_{protection}}{PV_{premium \text{ (for 1 bps spread)}}}$$

A useful market heuristic connects the spread, the hazard rate, and the Recovery Rate $R$:
$$s \approx \lambda (1 - R)$$

In [ ]:
class CDSPricer:
    def __init__(self, hazard_rate, recovery_rate, interest_rate, maturity, payment_freq=0.25):
        self.h = hazard_rate
        self.R = recovery_rate
        self.r = interest_rate
        self.T = maturity
        self.dt = payment_freq
        self.times = np.arange(self.dt, self.T + self.dt, self.dt)

    def df(self, t):
        """Discount factor"""
        return np.exp(-self.r * t)

    def survival_prob(self, t):
        """Survival probability under constant hazard rate"""
        return np.exp(-self.h * t)

    def compute_fair_spread(self):
        premium_leg = 0.0
        protection_leg = 0.0
        
        t_prev = 0.0
        s_prev = 1.0
        
        for t in self.times:
            df_t = self.df(t)
            s_curr = self.survival_prob(t)
            
            # Premium Leg: Payments on survival + accrued on default
            premium_leg += self.dt * df_t * s_curr
            premium_leg += 0.5 * self.dt * df_t * (s_prev - s_curr) 
            
            # Protection Leg: Payout upon default
            protection_leg += (1 - self.R) * df_t * (s_prev - s_curr)
            
            t_prev = t
            s_prev = s_curr
            
        return protection_leg / premium_leg

# Application Example
cds = CDSPricer(hazard_rate=0.02, recovery_rate=0.40, interest_rate=0.05, maturity=5.0)
fair_spread = cds.compute_fair_spread()
print(f"Fair CDS Spread: {fair_spread * 10000:.2f} bps")

## 4. Credit Valuation Adjustment (CVA)

CVA represents the market value of counterparty credit risk. It is the difference between the risk-free portfolio value and the true value considering the possibility of the counterparty's default. Mathematically, it is the risk-neutral expectation of the discounted loss over the life of a portfolio:

$$CVA = (1-R) \int_0^T \mathbb{E}^\mathbb{Q}\left[ \max(V(t), 0) \mid \tau = t \right] D(0, t) d\mathbb{P}(\tau \le t)$$

Assuming independence between the exposure $V(t)$ and the default time $\tau$ (i.e., ignoring Wrong-Way Risk), the expected exposure $EE(t) = \mathbb{E}[\max(V(t), 0)]$ decouples from the default probability. By discretizing the integral, where $S(t)$ is the survival probability and marginal default probability is $S(t_{i-1}) - S(t_i)$:

$$CVA \approx (1-R) \sum_{i=1}^m EE(t_i) \cdot D(0, t_i) \cdot \left[ S(0, t_{i-1}) - S(0, t_i) \right]$$

In [ ]:
def calculate_cva(exposures, times, hazard_rate, recovery_rate, interest_rate):
    """
    Calculates CVA for a given deterministic Expected Exposure (EE) profile.
    """
    cva = 0.0
    s_prev = 1.0
    
    for i in range(1, len(times)):
        t_curr = times[i]
        
        df = np.exp(-interest_rate * t_curr)
        s_curr = np.exp(-hazard_rate * t_curr)
        
        # Marginal probability of default in the interval
        marginal_pd = s_prev - s_curr
        
        # Average exposure in the interval (Trapezoidal approximation)
        ee_avg = (exposures[i-1] + exposures[i]) / 2.0
        
        cva += (1 - recovery_rate) * ee_avg * df * marginal_pd
        s_prev = s_curr
        
    return cva

# Application Example: Amortizing Loan / Derivative Profile
times = np.linspace(0, 5, 21) # Quarterly evaluation over 5 years
# Simulated exposure: starts at 100, amortizes linearly to 0
ee_profile = np.maximum(100 - 20 * times, 0) 

cva_value = calculate_cva(exposures=ee_profile, times=times, hazard_rate=0.02, recovery_rate=0.40, interest_rate=0.05)
print(f"Total Credit Valuation Adjustment (CVA): ${cva_value:.4f}")